In [1]:
import pandas as pd
import numpy as np

## Detecting encoding of datset

In [2]:
import chardet
path="/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv"
with open(path, "rb") as f:
    result=chardet.detect(f.read())
print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


In [3]:
df=pd.read_csv("/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv", encoding="cp1252")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [5]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


In [6]:
df.duplicated().sum()

np.int64(0)

> The dataset has no duplicate rows, no missing values

## correcting date format and adding some features

In [7]:
df["Order Date"]=pd.to_datetime(df["Order Date"])
df["Ship Date"]=pd.to_datetime(df["Ship Date"])
df["Order Month"]=df["Order Date"].dt.month
df["Ship Month"]=df["Ship Date"].dt.month
df["Order Year"]=df["Order Date"].dt.year
df["Ship Year"]=df["Ship Date"].dt.year

In [8]:
len(df[df["Order Date"].dt.month == 1].index)

381

In [9]:
month_sales=pd.DataFrame(columns=["Month", "Number of sales", "Total sales", "Total profit"])

In [10]:
month_sales= df.groupby("Order Month", sort=True).agg(
    Number_of_sales=("Sales","count"),
    Total_sales=("Sales","sum"),
    Total_profit=("Profit","sum")
).reset_index()
month_sales.rename(columns={"Order Month": "Month"}, inplace=True)
month_sales["Month"]=pd.to_datetime(month_sales["Month"],format="%m").dt.month_name()
month_sales

,Month,Number_of_sales,Total_sales,Total_profit
0,January,381,94924.8356,9134.4461
1,February,300,59751.2514,10294.6107
2,March,696,205005.4888,28594.6872
3,April,668,137762.1286,11587.4363
4,May,735,155028.8117,22411.3078
5,June,717,152718.6793,21285.7954
6,July,710,147238.0970,13832.6648
7,August,706,159044.0630,21776.9384
8,September,1383,307649.9457,36857.4753
9,October,819,200322.9847,31784.0413


## Finding KPI's

In [22]:
total_sale=month_sales["Total_sales"].sum()
total_profit=month_sales["Total_profit"].sum()
total_uni_orders=df["Order ID"].nunique()
avg_order_val=total_sale/total_uni_orders
profit_margin=(total_profit/total_sale)*100

> add discount in it

In [26]:
cat_sales=df.groupby("Category")[["Sales","Profit"]].sum().reset_index()
sub_cat_sales=df.groupby("Sub-Category")[["Sales","Profit"]].sum().reset_index()
region_sales=df.groupby("Region")[["Sales","Profit"]].sum().reset_index()
seg_sales=df.groupby("Segment")[["Sales","Profit"]].sum().reset_index()

In [38]:
sub_cat_sales

,Sub-Category,Sales,Profit
0,Accessories,167380.3180,41936.6357
1,Appliances,107532.1610,18138.0054
2,Art,27118.7920,6527.7870
3,Binders,203412.7330,30221.7633
4,Bookcases,114879.9963,-3472.5560
5,Chairs,328449.1030,26590.1663
6,Copiers,149528.0300,55617.8249
7,Envelopes,16476.4020,6964.1767
8,Fasteners,3024.2800,949.5182
9,Furnishings,91705.1640,13059.1436


In [28]:
region_sales

,Region,Sales,Profit
0,Central,501239.8908,39706.3625
1,East,678781.2400,91522.7800
2,South,391721.9050,46749.4303
3,West,725457.8245,108418.4489


In [29]:
df[['Sales','Profit','Discount']].corr()

,Sales,Profit,Discount
Sales,1.000000,0.479064,-0.028190
Profit,0.479064,1.000000,-0.219487
Discount,-0.028190,-0.219487,1.000000


In [37]:
loss_products = df[df['Profit'] < 0]
loss_by_subcat = loss_products.groupby('Sub-Category')['Profit'].sum()
loss_by_subcat

Sub-Category
Accessories     -930.6265
Appliances     -8629.6412
Binders       -38510.4964
Bookcases     -12152.2060
Chairs         -9880.8413
Fasteners        -33.1952
Furnishings    -6490.9134
Machines      -30118.6682
Phones         -7530.6235
Storage        -6426.3038
Supplies       -3015.6219
Tables        -32412.1483
Name: Profit, dtype: float64